# Tema 1 — Explorarea corpusului și primul prompt
În acest notebook vei explora corpusul curățat de comentarii YouTube și vei testa un prim prompt exploratoriu.

Vei testa 10 comentarii și vei reflecta asupra unor probleme precum ambiguitatea, țintele multiple, sarcasmul și confuzia dintre sentiment și poziționarea față de țintă.

## 1. Pregătire
Încărcăm bibliotecile necesare și cheia API pentru Gemini.
Modificați doar celula de configurare a studentului.

In [1]:
from pathlib import Path
import os
import json
import random
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

In [2]:
ROOT = Path.cwd()
while not (ROOT / ".env").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
load_dotenv(ROOT / ".env")

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
print("Root project:", ROOT)
print("Gemini key found:", GEMINI_API_KEY is not None)

Root project: c:\Users\admin\Desktop\Amalgamul\Facultate\Anul 2 - Sem 2\Modul 8 - Inginerie AI\echochamber-project-team-5
Gemini key found: True


## 2. Configurare
Modificați  această celulă.
Schimbați `student_id` cu folderul vostru: `student_01`, `student_02`, etc.

In [3]:
student_id = "student_03"
model = "gemini-2.5-flash"
temperature = 0.2
corpus_file = ROOT / "data" / "cleaned" / "corpus_youtube_large_clean.jsonl"
output_file = ROOT / "outputs" / f"{student_id}_prompt_outputs.jsonl"

## 3. Încărcăm corpusul curățat
Corpusul este salvat în format JSONL.
JSONL înseamnă: un comentariu pe fiecare linie.

In [4]:
# Citim fiecare linie din fișierul JSONL si o transformăm într-un dataframe pentru explorare
records = []
with corpus_file.open("r", encoding="utf-8") as f:
    for line in f:
        records.append(json.loads(line))
# Transformăm lista într-un DataFrame pentru explorare mai ușoară
df = pd.DataFrame(records)
df.head()

,id,source_platform,source_channel,text,text_raw,bubble_label,bubble_self_identified,topic,rhetoric_type,video_id,video_title,video_date,comment_date,likes,lang,collected_at
0,yt_5rHoTX3U_3Q_UgxaV5so7vyeXpyy8up4AaABAg,youtube,georgesimionoficial,Felicitării George Simion Președintele Românie...,Felicitării George Simion Președintele Românie...,None,False,None,None,5rHoTX3U_3Q,Turul României: realități de la firul ierbii,2025-09-30,2025-10-23,1,ro,2026-03-22
1,yt_5rHoTX3U_3Q_UgwJYiRLMbLfl2AipVR4AaABAg,youtube,georgesimionoficial,Asa trebuie să fiți printre oameni nu sa se do...,Asa trebuie să fiți printre oameni nu sa se do...,None,False,None,None,5rHoTX3U_3Q,Turul României: realități de la firul ierbii,2025-09-30,2025-10-02,5,ro,2026-03-22
2,yt_5rHoTX3U_3Q_UgzXqOk_SypZcQS-JcF4AaABAg,youtube,georgesimionoficial,Eu am votat cu George Simion din primul tur pt...,Eu am votat cu George Simion din primul tur pt...,None,False,None,None,5rHoTX3U_3Q,Turul României: realități de la firul ierbii,2025-09-30,2025-10-01,30,ro,2026-03-22
3,yt_5rHoTX3U_3Q_UgzpKghDX0_l-Gc3P4V4AaABAg,youtube,georgesimionoficial,Si de trebuie deposite de combustibil degeaba ...,Si de trebuie deposite de combustibil degeaba ...,None,False,None,None,5rHoTX3U_3Q,Turul României: realități de la firul ierbii,2025-09-30,2025-10-16,3,ro,2026-03-22
4,yt_5rHoTX3U_3Q_UgwqOTRSHNPj9cuGwNt4AaABAg,youtube,georgesimionoficial,Nu te descuraja că dobitoci și proști vor fi p...,Nu te descuraja că dobitoci și proști vor fi...,None,False,None,None,5rHoTX3U_3Q,Turul României: realități de la firul ierbii,2025-09-30,2025-10-01,9,ro,2026-03-22


In [5]:
print("Number of comments:", len(df))
print("Columns:", list(df.columns))

Number of comments: 30451
Columns: ['id', 'source_platform', 'source_channel', 'text', 'text_raw', 'bubble_label', 'bubble_self_identified', 'topic', 'rhetoric_type', 'video_id', 'video_title', 'video_date', 'comment_date', 'likes', 'lang', 'collected_at']


## 4. Explorare rapidă a corpusului
Ne uităm la canalele principale și la câteva exemple de comentarii.
Această etapă ne ajută să înțelegem ce tip de date avem înainte să folosim modelul.

In [6]:
# cele mai frecvente 15 canale sursă din dataset
df["source_channel"]# completează pentru a vedea cele mai frecvente 15 canale sursă din dataset

0        georgesimionoficial
1        georgesimionoficial
2        georgesimionoficial
3        georgesimionoficial
4        georgesimionoficial
                ...         
30446         Canal33Romania
30447         Canal33Romania
30448         Canal33Romania
30449         Canal33Romania
30450         Canal33Romania
Name: source_channel, Length: 30451, dtype: str

In [7]:
# aruncă o privire asupra unor comentarii random din dataset
df[["source_channel", "video_title", "text"]].sample(5, random_state=42)

,source_channel,video_title,text
23002,CălinGeorgescu-CanalulOficial,Călin Georgescu - Pacea de la București ( IPJ ...,Multă sănătate dl. Președinte Călin Georgescu....
5815,@CălinGeorgescu-CanalulOficial,Călin Georgescu - De ce vorbim despre Eminescu...,Un discurs care trebuia sa vina de la Cotrocen...
11191,RecorderRomania,Primarul Negoiță a construit șosele peste magi...,Autoritatile abilitate sa intervina!!! De acee...
11316,RecorderRomania,Primarul Negoiță a construit șosele peste magi...,In acest moment mai putem spune doar Dumnezeu ...
12505,RecorderRomania,DOCUMENTAR RECORDER. Singuri,E dureros.. e crunt.. simt vinovatie si recuno...


## 5. Alegem 10 comentarii pentru testarea promptului
Folosim 10 comentarii curate.
Puteți păstra eșantionarea aleatorie sau puteți selecta manual comentarii mai interesante.

In [8]:
sample_df = df.sample(10, random_state=42).copy()
sample_df[["source_channel", "text"]]

,source_channel,text
23002,CălinGeorgescu-CanalulOficial,Multă sănătate dl. Președinte Călin Georgescu....
5815,@CălinGeorgescu-CanalulOficial,Un discurs care trebuia sa vina de la Cotrocen...
11191,RecorderRomania,Autoritatile abilitate sa intervina!!! De acee...
11316,RecorderRomania,In acest moment mai putem spune doar Dumnezeu ...
12505,RecorderRomania,E dureros.. e crunt.. simt vinovatie si recuno...
9644,RecorderRomania,Cite dosare ați judecat și nu ați recuperat ni...
23843,turcescu111,"Totul duce către: Noua Ordine Mondială, pentru..."
11605,RecorderRomania,"Un hot corupt arogant si nesimtit, caruia nime..."
15486,RecorderRomania,"4:30 și încă 1% rămas pentru Crin Alcoolescu, ..."
7767,RecorderRomania,Vă mai dau niște firme din Galați care au alți...


Optional , poti alege sa folosesti  alta metoda de esantionare sau sa filtrezi dupa anumite canale sursa sau alte criterii. Important e sa ai un set de date mic pe care sa testezi promptul inainte de a-l rula pe intregul dataset.

## 6. Primul prompt exploratoriu
Completăm un prompt simplu pentru analizarea comentariilor politice.
Promptul trebuie să ceară:
- ținta comentariului;
- poziționarea față de țintă;
- tonul;
- tema;
- problema de interpretare;
- o justificare scurtă.
Important: tonul sau sentimentul general nu este același lucru cu poziționarea față de țintă.

In [32]:
SYSTEM_PROMPT = """
Ești un expert în analiză politică. Poziția ta este anti-suveranistă.
Analizează comentariul și extrage cele 7 dimensiuni sub formă de JSON valid.
"""

USER_PROMPT_TEMPLATE = """
Citește următorul comentariu și identifică:
1. target, 2. stance, 3. sentiment, 4. tone, 5. topic, 6. interpretation_problem, 7. echo_chamber_risk.
Returnează EXCLUSIV un obiect JSON valid.
Comentariu:
<<< {comment_text} >>>
"""

## 7. Conectarea la model
Folosim Gemini prin endpoint compatibil cu OpenAI.
Modelul și temperatura au fost setate mai sus.

In [33]:
from openai import OpenAI
client = OpenAI(
    api_key=GEMINI_API_KEY,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [34]:
def annotate_comment(comment_text):
    prompt = USER_PROMPT_TEMPLATE.format(comment_text=comment_text)
    response = client.chat.completions.create(
        model=model,
        temperature=temperature,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ]
    )
    return response.choices[0].message.content

## 8. Rulăm promptul pe 10 comentarii
Trimitem fiecare comentariu selectat la model și salvăm răspunsurile.

In [35]:
n_comments = 10  # schimbă aici: 3, 5 sau 10
sample_for_prompt = pd.read_json('../../data/cleaned/student_03_youtube_clean.jsonl', lines=True).head(n_comments)

outputs = []
for _, row in sample_for_prompt.iterrows():
    outputs.append({
        "source_channel": row.get("source_channel", ""),
        "video_title": row.get("video_title", ""),
        "comment_text": row["text"],
        "model_output": annotate_comment(row["text"])
    })
results_df = pd.DataFrame(outputs)
results_df

,source_channel,video_title,comment_text,model_output
0,StareaNatiei,Workshop cu Gabriel Liiceanu și Dragoș Pătraru...,"În limba bătrână ""să mă hăituiască"" 😉. ""Am glu...","```json\n{\n ""target"": ""Potențiali critici sa..."
1,StareaNatiei,Workshop cu Gabriel Liiceanu și Dragoș Pătraru...,"Dacă considerați că Ilie Bolojan este MOISE, c...","```json\n{\n ""target"": ""Publicul care ar pute..."
2,StareaNatiei,Workshop cu Gabriel Liiceanu și Dragoș Pătraru...,29:34 Am si eu impresia ca genul acesta de ati...,"```json\n{\n ""target"": ""Domnul Gabriel Liicea..."
3,StareaNatiei,Workshop cu Gabriel Liiceanu și Dragoș Pătraru...,"Pacat acest ""rezumat"" a omis fix partile cu ad...","```json\n{\n ""target"": ""Un 'rezumat' (sumar) ..."
4,StareaNatiei,Workshop cu Gabriel Liiceanu și Dragoș Pătraru...,"""Cum să rămânem oameni într-o lume condusă de ...","```json\n{\n ""target"": ""Comentariul vizează d..."
5,StareaNatiei,Workshop cu Gabriel Liiceanu și Dragoș Pătraru...,E prima oara cand aud un om care vorbeste doar...,"```json\n{\n ""target"": ""Vorbitorul din înregi..."


# 9. Verificam rezultatele

In [36]:
results_df.model_output[0]

'```json\n{\n  "target": "Potențiali critici sau adversari ideologici (probabil suveraniști), cărora autorul le adresează o provocare sau o anticipare a reacției lor la o afirmație anterioară anti-suveranistă. Comentariul este o reconfirmare a unei poziții care, prin natura sa, este susceptibilă de a genera opoziție din partea cercurilor naționaliste sau tradiționaliste.",\n  "stance": "Defensivă, dar și provocatoare și asertivă. Autorul își reconfirmă o poziție anterioară (presupus anti-suveranistă), anticipând critici, dar afirmând că \'adevărul\' a fost rostit, chiar și sub formă de \'glumă\'. Este o poziție de sfidare față de o potențială \'hăituire\' din partea taberei opuse, sugerând o conștientizare a impactului anti-suveranist al afirmației inițiale.",\n  "sentiment": "Încrezător, sfidător, ușor triumfător, dar și cu o notă de anticipare a conflictului. Emoțiile sunt pozitive în raport cu propria afirmație și cu reacția la aceasta, denotând o satisfacție în a provoca și a susți

In [37]:
# funcție pentru a curăța și parsa output-ul modelului, care poate conține JSON în diferite formate (text simplu sau bloc ```json)

def parse_model_output(text):
    # Modelul poate întoarce JSON ca text simplu sau în bloc ```json
    text = text.replace("```json", "")
    text = text.replace("```", "")
    text = text.strip()
    
    return json.loads(text)

In [38]:
print(results_df["model_output"].isna().sum(), "rows are missing model outputs.")

0 rows are missing model outputs.


In [39]:
import pandas as pd

for _, row in results_df.iterrows():
    # This catches both None and NaN values
    if pd.isna(row["model_output"]):
        continue

    parsed = parse_model_output(row["model_output"])
    # ... rest of your code

In [40]:
parsed_outputs = []

for _, row in results_df.iterrows():
    
    # 1. Verificăm dacă model_output e gol în DataFrame
    if pd.isna(row["model_output"]) or row["model_output"] is None:
        continue
        
    parsed = parse_model_output(row["model_output"])
    
    # 2. SIGURANȚĂ: Dacă parserul a eșuat și a returnat None, îi dăm un dicționar gol
    if not isinstance(parsed, dict):
        parsed = {} # Previne AttributeError
        
    parsed_outputs.append({
        "source_channel": row["source_channel"],
        "video_title": row["video_title"],
        "comment_text": row["comment_text"],
        "target": parsed.get("target", ""),
        "stance": parsed.get("stance", ""),
        "sentiment": parsed.get("sentiment", ""),
        "tone": parsed.get("tone", ""),
        "topic": parsed.get("topic", ""),
        "interpretation_problem": parsed.get("interpretation_problem", ""),
        "reason": parsed.get("reason", "") # Ai grijă, în promptul tău anterior cheia se numea "echo_chamber_risk", nu "reason"!
    })

# 10 Salvarea csv si inspectarea rezulatelor
- salvati ca csv 
- deschideti csv si verificati rezultatele 
- raspundeti la urmatoarele intrebare: promptul separă corect sentimentul general de poziționarea față de target? 

In [41]:
import pandas as pd

# Presupunem că 'parsed_outputs' este lista ta de rezultate procesate
# Dacă ai deja un DataFrame numit results_df, îl poți folosi direct pe acela
df_final = pd.DataFrame(parsed_outputs)

# Definirea numelui fișierului
nume_fisier = "student_3_tema1_first_prompt.csv"

# Salvarea propriu-zisă
df_final.to_csv(nume_fisier, index=False, encoding='utf-8-sig')

print(f"Fișierul a fost salvat cu succes: {nume_fisier}")

Fișierul a fost salvat cu succes: student_3_tema1_first_prompt.csv


1. Unde a funcționat bine promptul?

    La identificarea nuanțelor politice: A înțeles perfect ideologia (a taxat imediat „cultul personalității”, „resemnarea civică” și „populismul”).

2. Unde a eșuat?

    La formatare: A ignorat complet structura scurtă de JSON și a scris eseuri lungi în fiecare celulă, în loc de etichete simple.

    A inventat categorii: La tone și stance a scris text liber în loc să aleagă din listă.

3. A confundat sentimentul cu stance-ul?

    Da. La comentariul 2, pentru că textul era agresiv și alarmist (Sentiment Negativ), modelul a inventat eticheta "Anti-glorificare a liderilor" în loc să spună clar dacă e Pro sau Anti-suveranist.

4. Au creat probleme sarcasmul, ambiguitatea sau țintele multiple?

    Da. În loc să tranșeze problema, modelul s-a plâns în text de lipsa de context și de emoji-uri (la comentariile 1, 3 și 6), blocându-se în detalii.
5. Ce modifici în versiunea următoare?
    Îi vom da un exemplu scurt de răspuns, îi interzicem explicațiile lungi, simplificăm overall promptu